In [4]:
# ============================
# Instalação e montagem do Drive
# ============================
!pip install aiohttp nest_asyncio pandas tqdm > /dev/null

from google.colab import drive
drive.mount('/content/drive')

# ============================
# Imports
# ============================
import requests, re, aiohttp, asyncio, nest_asyncio, pandas as pd, json, os
from bs4 import BeautifulSoup
from tqdm import tqdm
from datetime import datetime
import time

nest_asyncio.apply()

# ============================
# Função para limpar HTML
# ============================
def limpar_html(html):
    if not html:
        return ""
    soup = BeautifulSoup(html, "html.parser")
    return soup.get_text(separator=" ", strip=True)

# ============================
# Configurações
# ============================
base_url = "https://www.promofarma.com.br"
headers = {"User-Agent": "Mozilla/5.0"}

max_retries_page = 3
sleep_time = 5
max_paginas = 60  # limitador

# Caminhos
data_nome = datetime.now().strftime("%Y-%m-%d")
base_drive = f"/content/drive/MyDrive/WebScraping/FEITO/PROMOFARMA/COLETA_PROMOFARMA/{data_nome}"
pasta_imagens = os.path.join(base_drive, "imagens")
pasta_parciais = os.path.join(base_drive, "parciais")
pasta_final = os.path.join(base_drive, "FINAL")

os.makedirs(pasta_imagens, exist_ok=True)
os.makedirs(pasta_parciais, exist_ok=True)
os.makedirs(pasta_final, exist_ok=True)

drive_json_path = os.path.join(base_drive, "promofarma_api_urls.json")
os.makedirs(os.path.dirname(drive_json_path), exist_ok=True)

# ============================
# Categorias
# ============================
categorias_dict = {
    1: ("Todas as categorias", [
        "/medicamentos",
        "/beleza-e-higiene",
        "/saude-e-bem-estar",
        "/mamae-e-bebe",
        "/vitaminas-e-suplementos"
    ]),
    2: ("Medicamentos", ["/medicamentos"]),
    3: ("Beleza e Higiene", ["/beleza-e-higiene"]),
    4: ("Saúde e Bem Estar", ["/saude-e-bem-estar"]),
    5: ("Mamãe e Bebê", ["/mamae-e-bebe"]),
    6: ("Vitaminas e Suplementos", ["/vitaminas-e-suplementos"])
}

# ============================
# Menu categorias
# ============================
print("Escolha a categoria para coleta:")
for key, value in categorias_dict.items():
    print(f"[{key}] {value[0]}")

while True:
    try:
        escolha = int(input("Digite o número da opção desejada: "))
        if escolha in categorias_dict:
            break
        else:
            print("Opção inválida. Tente novamente.")
    except:
        print("Digite um número válido.")

categorias = categorias_dict[escolha][1]
print(f"\n✅ Você selecionou: {categorias_dict[escolha][0]}\n", flush=True)

# ============================
# Pergunta quantidade
# ============================
while True:
    print("Deseja coletar:")
    print("[1] Todos os produtos")
    print("[2] Quantidade específica")
    opcao = input("Digite 1 ou 2: ")

    if opcao == "1":
        qtd_por_categoria = None
        print("\n✅ O código coletará todos os produtos disponíveis.\n", flush=True)
        break
    elif opcao == "2":
        try:
            qtd_por_categoria = int(input("Informe a quantidade de produtos a coletar por categoria: "))
            if qtd_por_categoria > 0:
                print(f"\n✅ O código coletará até {qtd_por_categoria} produtos por categoria.\n", flush=True)
                break
            else:
                print("Digite um número maior que zero.")
        except:
            print("Digite um número válido.")
    else:
        print("Opção inválida. Digite 1 ou 2.")

# ============================
# Funções
# ============================
def coletar_links_categoria(url, limite):
    links = []
    page = 1
    while (limite is None or len(links) < limite) and page <= max_paginas:
        success = False
        for attempt in range(max_retries_page):
            print(f"🔎 Página {page} (tentativa {attempt+1})", flush=True)
            try:
                resp = requests.get(f"{url}?page={page}", headers=headers, timeout=10)
                if resp.status_code != 200:
                    time.sleep(2)
                    continue

                soup = BeautifulSoup(resp.text, "html.parser")
                page_links = [a["href"] for a in soup.select("a[href*='/p']")]

                if page_links:
                    novos_links = [l for l in page_links if l not in links]
                    if novos_links:
                        for link in novos_links:
                            if limite is None or len(links) < limite:
                                links.append(link)

                        # ======= PRINT NO FORMATO DESEJADO =======
                        print(f"✅ {len(novos_links)} links coletados na página {page}")
                        for l in novos_links:
                            full_link = l if l.startswith("https://") else base_url + l
                            print(f"    {full_link}")
                        # =========================================

                        success = True
                break
            except Exception as e:
                print(f"⚠️ Erro: {e}", flush=True)
            time.sleep(sleep_time)
        page += 1
    return list(set(links))


def extrair_product_id(produto_url, max_retries=3):
    for attempt in range(max_retries):
        try:
            resp = requests.get(produto_url, headers=headers, timeout=10)
            if resp.status_code != 200:
                time.sleep(1)
                continue
            match = re.search(r'"productId":"(\d+)"', resp.text)
            if match:
                return match.group(1)
        except:
            time.sleep(1)
    return None


async def fetch(session, url):
    try:
        async with session.get(url, headers=headers) as resp:
            if resp.status != 200:
                return None
            return await resp.json()
    except:
        return None


async def baixar_produtos(api_urls):
    produtos = []
    async with aiohttp.ClientSession() as session:
        tasks = [fetch(session, url) for url in api_urls]
        for f in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
            result = await f
            if result:
                produtos.extend(result)
    return produtos

# ============================
# Ler APIs antigas
# ============================
if os.path.exists(drive_json_path):
    with open(drive_json_path, "r") as f:
        saved_api_urls = json.load(f)
else:
    saved_api_urls = []

# ============================
# Coleta
# ============================
todos_dados_iniciais = []

for cat in categorias:
    print(f"\n=== Coletando categoria: {cat} ===", flush=True)
    links = coletar_links_categoria(base_url + cat, qtd_por_categoria)
    print(f"\n✅ Total de links coletados: {len(links)}\n", flush=True)

    new_product_ids = []
    for link in links:
        url = base_url + link if link.startswith("/") else link
        pid = extrair_product_id(url)
        if pid:
            api_url = f"{base_url}/api/catalog_system/pub/products/search?fq=productId:{pid}"
            if api_url not in saved_api_urls:
                saved_api_urls.append(api_url)
            new_product_ids.append(pid)
        time.sleep(sleep_time)

    with open(drive_json_path, "w") as f:
        json.dump(saved_api_urls, f, indent=2)

    produtos_cat = asyncio.run(baixar_produtos(
        [f"{base_url}/api/catalog_system/pub/products/search?fq=productId:{pid}" for pid in new_product_ids]
    ))
    print(f"📦 Total de produtos coletados para {cat}: {len(produtos_cat)}", flush=True)

    for i, p in enumerate(produtos_cat, 1):
        try:
            item = p["items"][0]
            oferta = item["sellers"][0]["commertialOffer"]
            link_produto = p["link"]
            if not link_produto.startswith("https://www.promofarma.com.br"):
                link_produto = "https://www.promofarma.com.br" + link_produto
            dados = {
                "Código": p["productId"],
                "Nome": p["productName"],
                "Preço Promocional": oferta["Price"],
                "Descrição": limpar_html(p.get("description", "")),
                "Especificações": limpar_html(" | ".join([f"{prop['name']}: {', '.join(prop['values'])}" for prop in p.get("properties", [])])),
                "Imagem": item["images"][0]["imageUrl"] if item["images"] else "",
                "URL": link_produto,
                "Loja": "Promofarma",
            }
            todos_dados_iniciais.append(dados)
        except:
            continue

# ============================
# Salvamento parcial e download imagens
# ============================
todos = []
for i, produto in enumerate(todos_dados_iniciais, 1):
    try:
        codigo = produto.get("Código", f"produto_{i}")
        caminho_img = os.path.join(pasta_imagens, f"{codigo}.png")

        if not os.path.exists(caminho_img) and produto.get("Imagem") not in [None, "N/A"]:
            img = requests.get(produto['Imagem'], stream=True, timeout=10)
            if img.status_code == 200:
                with open(caminho_img, 'wb') as f:
                    for chunk in img.iter_content(1024):
                        f.write(chunk)

        todos.append(produto)

        if i % 100 == 0 or i == len(todos_dados_iniciais):
            df = pd.DataFrame(todos)
            colunas = ['Código', 'Nome', 'Preço Promocional', 'Descrição', 'Especificações', 'Imagem', 'URL', 'Loja']
            df = df[colunas]
            df.to_csv(os.path.join(pasta_parciais, f"promofarma_parcial_{i}.csv"), index=False, sep=";")
            df.to_excel(os.path.join(pasta_parciais, f"promofarma_parcial_{i}.xlsx"), index=False)
            print(f"💾 Dados salvos parcialmente com {i} produtos.", flush=True)
            todos.clear()
    except Exception as e:
        print(f"❌ Erro inesperado no produto {i}: {e}", flush=True)

# ============================
# Unificar final
# ============================
print("\n📁 Unificando arquivos CSV...", flush=True)
csvs = [f for f in os.listdir(pasta_parciais) if f.endswith(".csv")]
df_final = pd.DataFrame()

for csv in csvs:
    df = pd.read_csv(os.path.join(pasta_parciais, csv), sep=";")
    df_final = pd.concat([df_final, df], ignore_index=True)

csv_final = os.path.join(pasta_final, f"promofarma_FINAL_{data_nome}.csv")
excel_final = os.path.join(pasta_final, f"promofarma_FINAL_{data_nome}.xlsx")

df_final.to_csv(csv_final, index=False, sep=";")
df_final.to_excel(excel_final, index=False)

print(f"✅ CSV final salvo: {csv_final}", flush=True)
print(f"✅ Excel final salvo: {excel_final}", flush=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Escolha a categoria para coleta:
[1] Todas as categorias
[2] Medicamentos
[3] Beleza e Higiene
[4] Saúde e Bem Estar
[5] Mamãe e Bebê
[6] Vitaminas e Suplementos
Digite o número da opção desejada: 2

✅ Você selecionou: Medicamentos

Deseja coletar:
[1] Todos os produtos
[2] Quantidade específica
Digite 1 ou 2: 2
Informe a quantidade de produtos a coletar por categoria: 5

✅ O código coletará até 5 produtos por categoria.


=== Coletando categoria: /medicamentos ===
🔎 Página 1 (tentativa 1)
✅ 14 links coletados na página 1
    https://www.promofarma.com.br/naridrin-solucao-nasal-30ml-1037953/p
    https://www.promofarma.com.br/esomeprazol-magnesico-40mg-28-comprimidos-revestidos-ranbaxy-1047254/p
    https://www.promofarma.com.br/tadalafila-5mg-30-comprimidos-revestidos-eurofarma-1027804/p
    https://www.promofarma.com.br/rosuvastatina-calcica-10mg-30-comprim

100%|██████████| 5/5 [00:01<00:00,  4.40it/s]

📦 Total de produtos coletados para /medicamentos: 5


💾 Dados salvos com 5 produtos.

📁 Unificando arquivos CSV...
✅ CSV final salvo: /content/drive/MyDrive/WebScraping/FEITO/PROMOFARMA/COLETA_PROMOFARMA/2025-09-04/promofarma_FINAL_2025-09-04.csv
✅ Excel final salvo: /content/drive/MyDrive/WebScraping/FEITO/PROMOFARMA/COLETA_PROMOFARMA/2025-09-04/promofarma_FINAL_2025-09-04.xlsx
